In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data")

game_file = next(DATA_DIR.glob("game_data_public.*.csv.gz"))
replay_file = next(DATA_DIR.glob("replay_data_public.*.csv.gz"))

print("Game file:", game_file.name)
print("Replay file:", replay_file.name)

# Load a manageable sample first
GAME_KEY = ["draft_id", "match_number", "game_number"]

game_df = pd.read_csv(
    game_file,
    nrows=100,
)

wanted_keys = set(
    map(
        tuple,
        game_df[GAME_KEY].itertuples(index=False, name=None)
    )
)

replay_parts = []

for chunk in pd.read_csv(
    replay_file,
    chunksize=10_000,
    low_memory=False,
):
    keys = list(
        zip(
            chunk["draft_id"],
            chunk["match_number"],
            chunk["game_number"],
        )
    )

    mask = [key in wanted_keys for key in keys]

    if any(mask):
        replay_parts.append(chunk.loc[mask])

replay_df = pd.concat(
    replay_parts,
    ignore_index=True,
)

print("game rows:", len(game_df))
print("replay rows:", len(replay_df))

Game file: game_data_public.Cube_-_Powered.PremierDraft.csv.gz
Replay file: replay_data_public.Cube_-_Powered.PremierDraft.csv.gz
game rows: 100
replay rows: 100


In [3]:
import duckdb
import numpy as np
import pandas as pd

from gensim.models.doc2vec import Doc2Vec, TaggedDocument


DB_PATH = "../data/17lands.duckdb"

VECTOR_SIZE = 64
EPOCHS = 40
MIN_PAIR_PROB = 1e-5

# Converts continuous PPMI into token multiplicity for Doc2Vec.
# This is deliberately fairly conservative so very large PMI values
# do not completely dominate a card's document.
PPMI_SCALE = 5
MAX_REPEATS = 30


query = """
WITH

build_cards AS (
    SELECT
        db.draft_id,
        db.build_id,
        dbc.card_id,
        dbc.deck_count
    FROM deck_builds db
    JOIN deck_build_cards dbc
        ON dbc.build_id = db.build_id
    WHERE dbc.deck_count > 0
),

card_build_prob AS (
    SELECT
        bc.draft_id,
        bc.build_id,
        bc.card_id,

        1.0 - EXP(
            LGAMMA(40 - bc.deck_count + 1)
            - LGAMMA(11)
            - LGAMMA(40 - bc.deck_count - 10 + 1)
            - (
                LGAMMA(41)
                - LGAMMA(11)
                - LGAMMA(31)
            )
        ) AS p_seen

    FROM build_cards bc
),

build_pair_prob AS (
    SELECT
        a.draft_id,
        a.build_id,
        a.card_id AS card_a_id,
        b.card_id AS card_b_id,

        1.0

        - EXP(
            LGAMMA(40 - a.deck_count + 1)
            - LGAMMA(11)
            - LGAMMA(40 - a.deck_count - 10 + 1)
            - (
                LGAMMA(41)
                - LGAMMA(11)
                - LGAMMA(31)
            )
        )

        - EXP(
            LGAMMA(40 - b.deck_count + 1)
            - LGAMMA(11)
            - LGAMMA(40 - b.deck_count - 10 + 1)
            - (
                LGAMMA(41)
                - LGAMMA(11)
                - LGAMMA(31)
            )
        )

        + CASE
            WHEN 40 - a.deck_count - b.deck_count >= 10
            THEN EXP(
                LGAMMA(40 - a.deck_count - b.deck_count + 1)
                - LGAMMA(11)
                - LGAMMA(40 - a.deck_count - b.deck_count - 10 + 1)
                - (
                    LGAMMA(41)
                    - LGAMMA(11)
                    - LGAMMA(31)
                )
            )
            ELSE 0.0
        END AS p_seen_together

    FROM build_cards a
    JOIN build_cards b
        ON a.build_id = b.build_id
       AND a.card_id < b.card_id
),

draft_card_prob AS (
    SELECT
        draft_id,
        card_id,
        MAX(p_seen) AS p_seen
    FROM card_build_prob
    GROUP BY draft_id, card_id
),

draft_pair_prob AS (
    SELECT
        draft_id,
        card_a_id,
        card_b_id,
        MAX(p_seen_together) AS p_seen_together
    FROM build_pair_prob
    GROUP BY draft_id, card_a_id, card_b_id
),

draft_count AS (
    SELECT COUNT(DISTINCT draft_id)::DOUBLE AS n
    FROM deck_builds
),

card_prob AS (
    SELECT
        dcp.card_id,
        SUM(dcp.p_seen) / dc.n AS p_card
    FROM draft_card_prob dcp
    CROSS JOIN draft_count dc
    GROUP BY dcp.card_id, dc.n
),

pair_prob AS (
    SELECT
        dpp.card_a_id,
        dpp.card_b_id,
        SUM(dpp.p_seen_together) / dc.n AS p_ab
    FROM draft_pair_prob dpp
    CROSS JOIN draft_count dc
    GROUP BY dpp.card_a_id, dpp.card_b_id, dc.n
)

SELECT
    pp.card_a_id,
    pp.card_b_id,
    ca.card_name AS card_a,
    cb.card_name AS card_b,

    pp.p_ab,
    ap.p_card AS p_a,
    bp.p_card AS p_b,

    LN(
        pp.p_ab /
        (ap.p_card * bp.p_card)
    ) AS pmi

FROM pair_prob pp

JOIN card_prob ap
    ON ap.card_id = pp.card_a_id

JOIN card_prob bp
    ON bp.card_id = pp.card_b_id

JOIN cards ca
    ON ca.card_id = pp.card_a_id

JOIN cards cb
    ON cb.card_id = pp.card_b_id

WHERE
    pp.p_ab > ?
    AND ap.p_card > 0
    AND bp.p_card > 0
"""


# ------------------------------------------------------------------
# Calculate PMI
# ------------------------------------------------------------------

with duckdb.connect(DB_PATH, read_only=True) as con:
    pairs = con.execute(query, [MIN_PAIR_PROB]).df()


pairs["ppmi"] = pairs["pmi"].clip(lower=0)

pairs = pairs[
    np.isfinite(pairs["ppmi"])
    & (pairs["ppmi"] > 0)
].copy()

print(f"{len(pairs):,} positive-PMI card pairs")
print(f"{pairs['card_a_id'].nunique() + pairs['card_b_id'].nunique():,} raw card appearances")


# ------------------------------------------------------------------
# Construct one Doc2Vec document per card.
#
# Card A's document consists of cards positively associated with A.
# Stronger PPMI -> neighbour token occurs more often.
# ------------------------------------------------------------------

card_names = {}

neighbours = {}

for row in pairs.itertuples():
    a = str(row.card_a_id)
    b = str(row.card_b_id)

    card_names[a] = row.card_a
    card_names[b] = row.card_b

    repeats = max(
        1,
        min(
            MAX_REPEATS,
            int(round(row.ppmi * PPMI_SCALE)),
        ),
    )

    neighbours.setdefault(a, []).extend([b] * repeats)
    neighbours.setdefault(b, []).extend([a] * repeats)


documents = [
    TaggedDocument(
        words=tokens,
        tags=[card_id],
    )
    for card_id, tokens in neighbours.items()
    if tokens
]

print(f"{len(documents):,} card documents")


# ------------------------------------------------------------------
# Train a simple Doc2Vec model.
# ------------------------------------------------------------------

model = Doc2Vec(
    vector_size=VECTOR_SIZE,
    window=10,
    min_count=1,
    workers=4,
    epochs=EPOCHS,
    dm=1,
    negative=10,
    seed=42,
)

model.build_vocab(documents)
model.train(
    documents,
    total_examples=model.corpus_count,
    epochs=model.epochs,
)


# ------------------------------------------------------------------
# Convenience lookup
# ------------------------------------------------------------------

name_to_id = {
    name: card_id
    for card_id, name in card_names.items()
}


def similar_cards(card_name, n=10):
    card_id = name_to_id[card_name]

    results = model.dv.most_similar(
        card_id,
        topn=n,
    )

    return pd.DataFrame(
        [
            {
                "card": card_names[other_id],
                "similarity": similarity,
            }
            for other_id, similarity in results
        ]
    )


# Example:
similar_cards("Lightning Bolt", 15)

49,969 positive-PMI card pairs
1,082 raw card appearances
544 card documents


,card,similarity
0,Detective's Phoenix,0.696683
1,"Laelia, the Blade Reforged",0.685146
2,Goblin Rabblemaster,0.683143
3,Glorybringer,0.671410
4,Bonecrusher Giant,0.655569
5,"Kari Zev, Skyship Raider",0.647938
6,Monstrous Rage,0.642830
7,"Gut, True Soul Zealot",0.635023
8,Robber of the Rich,0.630112
9,Sorin of House Markov,0.627576


In [9]:
import numpy as np
import pandas as pd


def complete_deck_pairwise(seed_cards, topn=20, mode="mean"):
    seed_ids = [name_to_id[name] for name in seed_cards]

    seed_vecs = np.array([
        model.dv[card_id]
        for card_id in seed_ids
    ])

    seed_vecs /= np.linalg.norm(seed_vecs, axis=1, keepdims=True)

    excluded = set(seed_ids)
    rows = []

    for card_id in model.dv.index_to_key:
        if card_id in excluded:
            continue

        v = model.dv[card_id]
        v = v / np.linalg.norm(v)

        sims = seed_vecs @ v

        if mode == "mean":
            score = sims.mean()
        elif mode == "min":
            score = sims.min()
        elif mode == "harmonic":
            positive = np.clip(sims, 1e-6, None)
            score = len(positive) / np.sum(1.0 / positive)
        else:
            raise ValueError("mode must be mean, min, or harmonic")

        rows.append({
            "card": card_names[card_id],
            "score": score,
            "individual_similarities": sims,
        })

    return (
        pd.DataFrame(rows)
        .sort_values("score", ascending=False)
        .head(topn)
        .reset_index(drop=True)
    )


complete_deck_pairwise(
    [
        "Lightning Bolt",
        "Faerie Mastermind",
        "Monastery Swiftspear",
    ],
    topn=20,
    mode="min",
)

,card,score,individual_similarities
0,Arena of Glory,0.624752,"[0.62475157, 0.6570719, 0.7829896]"
1,Wrenn and Six,0.623470,"[0.62346995, 0.6354189, 0.68583053]"
2,Hellrider,0.614528,"[0.6145282, 0.6216613, 0.76377356]"
3,"Inti, Seneschal of the Sun",0.611853,"[0.6118535, 0.61488044, 0.70092165]"
4,"Nissa, Resurgent Animist",0.610842,"[0.61084217, 0.6325112, 0.7788469]"
5,Territorial Kavu,0.603484,"[0.6191165, 0.6034836, 0.64518154]"
6,Nova Hellkite,0.603127,"[0.60641134, 0.6031268, 0.7809999]"
7,Draconautics Engineer,0.602284,"[0.6022836, 0.68377554, 0.7717291]"
8,"Adeline, Resplendent Cathar",0.598843,"[0.5988426, 0.6401086, 0.7416102]"
9,Animate Dead,0.598001,"[0.5980005, 0.6047043, 0.74087596]"


In [15]:
import numpy as np
import pandas as pd


def complete_deck_ppmi(seed_cards, topn=20, min_seed_support=2):
    seed_ids = [name_to_id[name] for name in seed_cards]
    seed_idxs = [card_to_idx[card_id] for card_id in seed_ids]

    rows = []

    for idx, card_id in idx_to_card.items():
        if card_id in seed_ids:
            continue

        # Direct PPMI relationship between candidate and each seed.
        ppmi_scores = np.array([
            ppmi_matrix[idx, seed_idx]
            for seed_idx in seed_idxs
        ], dtype=float).ravel()

        support = np.count_nonzero(ppmi_scores > 0)

        if support < min_seed_support:
            continue

        # Geometric mean strongly rewards candidates with evidence
        # across multiple seeds and penalizes one-seed specialists.
        positive = np.clip(ppmi_scores, 1e-6, None)

        score = np.exp(np.mean(np.log(positive)))

        rows.append({
            "card": card_names[card_id],
            "score": score,
            "seed_support": support,
            **{
                f"ppmi_{seed}": value
                for seed, value in zip(seed_cards, ppmi_scores)
            },
        })

    return (
        pd.DataFrame(rows)
        .sort_values(
            ["seed_support", "score"],
            ascending=[False, False],
        )
        .head(topn)
        .reset_index(drop=True)
    )


complete_deck_ppmi(
    [
        "Lightning Bolt",
        "Faerie Mastermind",
        "Monastery Swiftspear",
        "Counterspell",
    ],
    topn=20,
    min_seed_support=2,
)

,card,score,seed_support,ppmi_Lightning Bolt,ppmi_Faerie Mastermind,ppmi_Monastery Swiftspear,ppmi_Counterspell
0,Fiery Islet,0.349245,4,0.373024,0.374045,0.471766,0.226013
1,Expressive Iteration,0.297495,4,0.172353,0.597189,0.135294,0.562485
2,Steam Vents,0.112481,4,0.106914,0.401697,0.122123,0.030521
3,Kolaghan's Command,0.100797,4,0.141222,0.280113,0.041362,0.063088
4,Enduring Curiosity,0.022446,3,0.000000,1.245569,0.356230,0.572112
5,Proft's Eidetic Memory,0.016155,3,0.000000,1.180202,0.076453,0.754959
6,Volcanic Island,0.015450,3,0.291678,0.404817,0.000000,0.482591
7,Spirebluff Canal,0.014585,3,0.000000,0.438103,0.150561,0.685966
8,Duelist of the Mind,0.013197,3,0.000000,1.317916,0.067564,0.340605
9,Galvanic Blast,0.011675,3,0.263825,0.360156,0.195512,0.000000


In [19]:
import numpy as np
import pandas as pd

from scipy.sparse import coo_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize


# Number of broad latent dimensions to REMOVE.
#
# Keep this fairly small. If this is too large, we start removing the
# interesting card-specific interactions that we're trying to discover.
CONTEXT_COMPONENTS = 12

# Dimensions of the final residual embedding.
RESIDUAL_COMPONENTS = 64


# ------------------------------------------------------------------
# 1. Learn the broad structure of the original PPMI matrix.
#
# This should capture large-scale effects such as:
#   - colour
#   - archetype
#   - aggro/control
#   - dominant format-specific shells
# ------------------------------------------------------------------

context_svd = TruncatedSVD(
    n_components=CONTEXT_COMPONENTS,
    random_state=42,
)

context_vectors = context_svd.fit_transform(ppmi_matrix)

print(
    f"Broad-context variance explained: "
    f"{context_svd.explained_variance_ratio_.sum():.3f}"
)


# ------------------------------------------------------------------
# 2. Calculate the residual for every observed PPMI edge.
#
# We avoid reconstructing the entire dense card x card matrix.
#
# For pair (i, j):
#
#     observed = PPMI(i, j)
#     expected = low-rank approximation(i, j)
#     residual = observed - expected
#
# Positive residual means:
#
#   "These two cards associate MORE strongly than their broad
#    deck/archetype contexts would predict."
# ------------------------------------------------------------------

residual_rows = []
residual_cols = []
residual_values = []

residual_records = []

components = context_svd.components_

for row in pairs.itertuples():

    if row.ppmi <= 0:
        continue

    i = card_to_idx[row.card_a_id]
    j = card_to_idx[row.card_b_id]

    # X_hat[i, j] = (U Sigma)[i] dot V^T[:, j]
    expected_ppmi = float(
        context_vectors[i] @ components[:, j]
    )

    residual = float(row.ppmi - expected_ppmi)

    residual_records.append({
        "card_a_id": row.card_a_id,
        "card_b_id": row.card_b_id,
        "card_a": row.card_a,
        "card_b": row.card_b,
        "ppmi": row.ppmi,
        "expected_ppmi": expected_ppmi,
        "residual": residual,
    })

    # For this experiment, only preserve unexpectedly STRONG
    # positive relationships.
    if residual > 0:
        residual_rows.extend([i, j])
        residual_cols.extend([j, i])
        residual_values.extend([residual, residual])


residual_pairs = pd.DataFrame(residual_records)

residual_matrix = coo_matrix(
    (
        residual_values,
        (residual_rows, residual_cols),
    ),
    shape=ppmi_matrix.shape,
).tocsr()


print(f"Original PPMI edges: {ppmi_matrix.nnz:,}")
print(f"Positive residual edges: {residual_matrix.nnz:,}")


# ------------------------------------------------------------------
# 3. Learn a new embedding from ONLY the unexplained relationships.
#
# This representation should be much less dominated by broad
# archetype membership.
# ------------------------------------------------------------------

residual_svd = TruncatedSVD(
    n_components=RESIDUAL_COMPONENTS,
    random_state=42,
)

residual_embeddings = residual_svd.fit_transform(
    residual_matrix
)

residual_embeddings = normalize(
    residual_embeddings
)

print(
    f"Residual embedding variance explained: "
    f"{residual_svd.explained_variance_ratio_.sum():.3f}"
)


# ------------------------------------------------------------------
# 4. Directly inspect what is unusual about a card.
#
# This is arguably the most important diagnostic for this experiment.
#
# It shows which pair relationships are stronger than the broad
# context model expected.
# ------------------------------------------------------------------

def unusual_partners(card_name, topn=20):
    card_id = name_to_id[card_name]

    mask = (
        (residual_pairs["card_a_id"] == card_id)
        | (residual_pairs["card_b_id"] == card_id)
    )

    x = residual_pairs.loc[mask].copy()

    x["other_card"] = np.where(
        x["card_a_id"] == card_id,
        x["card_b"],
        x["card_a"],
    )

    return (
        x[x["residual"] > 0]
        .sort_values("residual", ascending=False)
        [
            [
                "other_card",
                "ppmi",
                "expected_ppmi",
                "residual",
            ]
        ]
        .head(topn)
        .reset_index(drop=True)
    )


# Example:
unusual_partners("Underworld Breach", 20)

Broad-context variance explained: 0.820
Original PPMI edges: 99,938
Positive residual edges: 60,928
Residual embedding variance explained: 0.618


,other_card,ppmi,expected_ppmi,residual
0,Regrowth,0.954230,0.328529,0.625701
1,Brain Freeze,2.212022,1.670446,0.541576
2,Lion's Eye Diamond,2.112898,1.630278,0.482620
3,Enlightened Tutor,1.459156,0.995315,0.463841
4,Pyrite Spellbomb,1.060659,0.632914,0.427746
5,"Jace, Wielder of Mysteries",1.457894,1.135604,0.322290
6,Spirebluff Canal,0.942395,0.634903,0.307492
7,Consider,1.103326,0.802230,0.301096
8,Talisman of Conviction,0.630703,0.372701,0.258002
9,Dack Fayden,0.977126,0.729267,0.247859


In [22]:
def similar_by_residual_embedding(card_name, topn=20):
    card_id = name_to_id[card_name]
    idx = card_to_idx[card_id]

    similarities = (
        residual_embeddings
        @ residual_embeddings[idx]
    )

    order = np.argsort(-similarities)

    rows = []

    for other_idx in order:
        other_id = idx_to_card[other_idx]

        if other_id == card_id:
            continue

        rows.append({
            "card": card_names[other_id],
            "similarity": float(
                similarities[other_idx]
            ),
        })

        if len(rows) == topn:
            break

    return pd.DataFrame(rows)


similar_by_residual_embedding(
    "Underworld Breach",
    20,
)

,card,similarity
0,Brain Freeze,0.848067
1,Lion's Eye Diamond,0.746241
2,Counterspell,0.724197
3,Pinnacle Emissary,0.675918
4,Mox Opal,0.673990
5,Mystic Confluence,0.670503
6,Talisman of Creativity,0.669679
7,Stock Up,0.668904
8,Aether Spellbomb,0.658452
9,Gitaxian Probe,0.651601
